# Unified Advanced Perceptual Autoencoder (All Categories)

This notebook consolidates the best-performing anomaly detection architectural choices across the team into a single, unified pipeline.

### Core Advanced Features:
1. **Dynamic Augmentations**: Object vs Texture-specific transforms (Noise, Jitter, Rotations).
2. **Object-Aware Masked Image Modeling (MIM)**: Random patch zeroing applied *only* to the Otsu+Canny foreground (with morphological dilation) to prevent trivial identity mapping.
3. **Fair Evaluation & Callbacks**: Uses an 85/15 validation split, Early Stopping, Model Checkpointing, and ReduceLROnPlateau.
4. **ResNet-18 Perceptual Loss**: Uses a frozen PatchCore backbone to penalize differences in deep feature space.
5. **Ablation Study**: Evaluates the model with and without Perceptual Loss/MIM to empirically demonstrate their impact.
6. **Strict Metrics**: Outputs AUPIMO, Image AUROC, and full confusion matrix in accordance with project rules.



## 1. Imports and Setup
We import PyTorch, Anomalib metrics, and setup data paths.


In [7]:
import os
import sys
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.models as models
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import copy
from sklearn.metrics import precision_recall_curve, f1_score, confusion_matrix, roc_auc_score
from torchmetrics.functional.image import structural_similarity_index_measure as ssim
from anomalib.metrics import AUPIMO

# Import Fair Split from repository
from app.domain.data import build_fair_evaluation_split

print(f"PyTorch Version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = Path("data/raw/mvtec_ad") # Use standard project path
if not DATA_ROOT.exists():
    DATA_ROOT = Path("../../data/raw/mvtec_ad")
SIZE = 256
BATCH_SIZE = 16  # Reduced to 2 because another GPU process (PID 1495081) is hogging 2.3GB of VRAM
EPOCHS = 120



PyTorch Version: 2.5.1+cu121


## 2. Dynamic Augmentations & Object-Aware Masking (MIM)
We apply specific transforms for Textures vs Objects.
We also extract a morphologically dilated Otsu+Canny mask to isolate the object and apply Masked Image Modeling (MIM) exclusively to its foreground pixels.


In [8]:

TEXTURE_CATEGORIES = {'carpet', 'grid', 'leather', 'tile', 'wood'}

class AddGaussianNoise:
    def __init__(self, mean=0., std=0.05):
        self.std = std
        self.mean = mean
        
    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean

def get_transforms(category, is_train=True):
    if not is_train:
        return T.Compose([T.Resize((SIZE, SIZE)), T.ToTensor()])
        
    is_texture = category in TEXTURE_CATEGORIES
    
    if is_texture:
        # Texture Augmenter: Severe rotations, flips, noise
        return T.Compose([
            T.Resize((SIZE, SIZE)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(180),
            T.ColorJitter(brightness=0.1, contrast=0.1),
            T.ToTensor(),
            AddGaussianNoise(std=0.03)
        ])
    else:
        # Object Augmenter: Minor shifts, color jitter
        return T.Compose([
            T.Resize((SIZE, SIZE)),
            T.RandomAffine(degrees=10, translate=(0.05, 0.05)),
            T.ColorJitter(brightness=0.2, contrast=0.2),
            T.ToTensor()
        ])

def extract_otsu_canny_mask_dilated(img_np):
    """Extract object mask using Otsu + Canny, with morphological dilation to cover edges."""
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    edges = cv2.Canny(blurred, 50, 150)
    combined = cv2.bitwise_or(thresh, edges)
    
    # Morphological Dilation to encompass edge boundaries
    kernel = np.ones((7,7), np.uint8)
    dilated = cv2.dilate(combined, kernel, iterations=2)
    
    # Extract largest component
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(dilated, connectivity=8)
    if num_labels > 1:
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask = np.where(labels == largest_label, 255, 0).astype(np.uint8)
    else:
        mask = dilated
        
    return mask / 255.0

def apply_mim_mask(img_tensor, mask_tensor, patch_size=32, mask_ratio=0.25):
    """Apply Masked Image Modeling (MIM) by zeroing patches exclusively inside the object mask."""
    _, H, W = img_tensor.shape
    corrupted = img_tensor.clone()
    
    # Calculate valid foreground pixels
    valid_y, valid_x = torch.where(mask_tensor.squeeze() > 0)
    if len(valid_y) == 0: 
        return corrupted # Background only, no mask
        
    num_patches_total = len(valid_y) / (patch_size * patch_size)
    num_mask = int(num_patches_total * mask_ratio)
    
    if num_mask <= 0: return corrupted
    
    # Randomly select patch centers
    indices = torch.randperm(len(valid_y))[:num_mask]
    
    for idx in indices:
        cy, cx = valid_y[idx].item(), valid_x[idx].item()
        y1, y2 = max(0, cy - patch_size//2), min(H, cy + patch_size//2)
        x1, x2 = max(0, cx - patch_size//2), min(W, cx + patch_size//2)
        corrupted[:, y1:y2, x1:x2] = 0.0 # Zero out patch
        
    return corrupted

class MVTecDataset(Dataset):
    def __init__(self, paths, category, is_train=True, use_mim=False):
        self.paths = paths
        self.category = category
        self.is_train = is_train
        self.use_mim = use_mim
        self.transform = get_transforms(category, is_train)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img_np = np.array(img.resize((SIZE, SIZE)))
        
        mask = extract_otsu_canny_mask_dilated(img_np)
        mask_tensor = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        
        # Original clean image
        clean_tensor = self.transform(img)
        
        # Generate corrupted image for MIM
        if self.is_train and self.use_mim:
            corrupted_tensor = apply_mim_mask(clean_tensor, mask_tensor)
        else:
            corrupted_tensor = clean_tensor
            
        return corrupted_tensor, clean_tensor, mask_tensor

def load_category_paths(category):
    cat_dir = DATA_ROOT / category
    if not cat_dir.exists():
        raise FileNotFoundError(f"Category {category} not found at {cat_dir}")
        
    try:
        manifest = get_manifest(DATA_ROOT)
        split = build_fair_evaluation_split(manifest, category, seed=42)
        train_good = [Path(p) for p in split.fit_paths]
        val_good = [Path(p) for p in split.val_paths]
    except Exception:
        train_good_all = sorted((cat_dir / "train" / "good").glob("*.png"))
        np.random.seed(42)
        shuffled = np.random.permutation(train_good_all)
        split_idx = int(len(shuffled) * 0.85)
        train_good, val_good = list(shuffled[:split_idx]), list(shuffled[split_idx:])
    
    test_good = sorted((cat_dir / "test" / "good").glob("*.png"))
    test_defect, test_defect_masks = [], []
    test_dir = cat_dir / "test"
    if test_dir.exists():
        for defect_dir in sorted(test_dir.iterdir()):
            if not defect_dir.is_dir() or defect_dir.name == "good":
                continue
            for img_path in sorted(defect_dir.glob("*.png")):
                mask_path = cat_dir / "ground_truth" / defect_dir.name / f"{img_path.stem}_mask.png"
                test_defect.append(img_path)
                test_defect_masks.append(mask_path)

    return train_good, val_good, test_good, test_defect, test_defect_masks



## 3. Architecture & Tri-Fold Loss (Perceptual)


In [9]:
class ConvAutoencoder(nn.Module):
    def __init__(self, bottleneck_channels=32):
        super().__init__()
        # Encoder (ELU for smoother gradients)
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ELU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ELU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(128), nn.ELU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(256), nn.ELU(),
            nn.Conv2d(256, bottleneck_channels, kernel_size=3, stride=1, padding=1)
        )
        # Decoder
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(bottleneck_channels, 256, kernel_size=3, stride=1, padding=1), nn.BatchNorm2d(256), nn.ELU(),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(128), nn.ELU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ELU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ELU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.dec(self.enc(x))

class PerceptualLossModule(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).eval()
        for param in resnet.parameters():
            param.requires_grad = False
            
        self.layer1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool, resnet.layer1)
        self.layer2 = nn.Sequential(self.layer1, resnet.layer2)
        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        
    def forward(self, y_true, y_pred):
        y_true_norm = self.normalize(y_true)
        y_pred_norm = self.normalize(y_pred)
        return (F.mse_loss(self.layer1(y_pred_norm), self.layer1(y_true_norm)) + 
                F.mse_loss(self.layer2(y_pred_norm), self.layer2(y_true_norm))) / 2.0

def combined_loss_fn(y_pred, y_true, perceptual_module, use_perceptual=True, alpha=0.84, beta=0.16, gamma=0.05):
    loss_ssim = 1.0 - ssim(y_pred, y_true, data_range=1.0)
    loss_l1 = F.l1_loss(y_pred, y_true)
    
    if use_perceptual and perceptual_module is not None:
        loss_perceptual = perceptual_module(y_true, y_pred)
        return alpha * loss_ssim + beta * loss_l1 + gamma * loss_perceptual
    else:
        # Baseline: Just SSIM and L1
        return alpha * loss_ssim + (1-alpha) * loss_l1



## 4. Callbacks & Training Loop (With Validation)


In [10]:
class EarlyStoppingAndCheckpoint:
    def __init__(self, patience=7):
        self.patience = patience
        self.counter = 0
        self.best_loss = float('inf')
        self.best_state_dict = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_state_dict = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

def train_autoencoder(category, use_perceptual=True, use_mim=True):
    print(f"\n{'='*50}")
    print(f"Training {category} | Perceptual={use_perceptual} | MIM={use_mim}")
    
    torch.manual_seed(42)
    np.random.seed(42)
    
    try:
        train_g, val_g, _, _, _ = load_category_paths(category)
    except Exception as e:
        print(f"Error loading {category}: {e}")
        return None
        
    train_ds = MVTecDataset(train_g, category, is_train=True, use_mim=use_mim)
    val_ds = MVTecDataset(val_g, category, is_train=False, use_mim=False)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = ConvAutoencoder().to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
    
    perceptual_module = PerceptualLossModule().to(DEVICE) if use_perceptual else None
    callbacks = EarlyStoppingAndCheckpoint(patience=7)
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for corrupted_x, clean_x, mask in train_loader:
            corrupted_x, clean_x, mask = corrupted_x.to(DEVICE), clean_x.to(DEVICE), mask.to(DEVICE)
            
            optimizer.zero_grad()
            recon = model(corrupted_x)
            
            # Loss applied only to object foreground
            recon_masked = recon * mask
            clean_masked = clean_x * mask
            
            loss = combined_loss_fn(recon_masked, clean_masked, perceptual_module, use_perceptual)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * corrupted_x.size(0)
            
        train_loss /= len(train_ds)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for corrupted_x, clean_x, mask in val_loader:
                corrupted_x, clean_x, mask = corrupted_x.to(DEVICE), clean_x.to(DEVICE), mask.to(DEVICE)
                recon = model(corrupted_x)
                
                recon_masked = recon * mask
                clean_masked = clean_x * mask
                loss = combined_loss_fn(recon_masked, clean_masked, perceptual_module, use_perceptual)
                val_loss += loss.item() * corrupted_x.size(0)
                
        val_loss /= len(val_ds)
        
        print(f"Epoch {epoch+1:02d}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")
        
        scheduler.step(val_loss)
        callbacks(val_loss, model)
        if callbacks.early_stop:
            print("Early Stopping Triggered!")
            break
            
        torch.cuda.empty_cache()
            
    # Restore best model
    model.load_state_dict(callbacks.best_state_dict)
    print("Training Complete. Best Model Restored.")
    return model



## 5. Strict Evaluation (AUPIMO, AUROC, Confusion Matrix)


In [11]:
def evaluate_model_strict(model, category):
    model.eval()
    _, _, test_good, test_defect, test_defect_masks = load_category_paths(category)
    
    if not test_defect:
        return
        
    all_test_paths = test_good + test_defect
    y_true_image = [0] * len(test_good) + [1] * len(test_defect)
    
    test_ds = MVTecDataset(all_test_paths, category, is_train=False)
    
    anomaly_maps = []
    ground_truth_masks = []
    image_scores = []
    
    with torch.no_grad():
        for i, (corrupted_x, clean_x, mask) in enumerate(test_ds):
            x = corrupted_x.unsqueeze(0).to(DEVICE)
            recon = model(x)
            
            orig_np = clean_x.permute(1, 2, 0).cpu().numpy()
            recon_np = recon.squeeze(0).permute(1, 2, 0).cpu().numpy()
            
            # Pixel-wise MAE
            anomaly_map = np.mean(np.abs(orig_np - recon_np), axis=-1)
            
            # Multiply by Object Mask to ignore background noise
            anomaly_map = anomaly_map * mask.squeeze().numpy()
            anomaly_maps.append(anomaly_map)
            
            # Image score = max anomaly value in the foreground
            image_scores.append(np.max(anomaly_map))
            
            if i >= len(test_good):
                # Defect image
                mask_path = test_defect_masks[i - len(test_good)]
                if mask_path.exists():
                    gt = np.array(Image.open(mask_path).resize((SIZE, SIZE))) > 0
                else:
                    gt = np.zeros((SIZE, SIZE))
                ground_truth_masks.append(gt)
            else:
                ground_truth_masks.append(np.zeros((SIZE, SIZE)))
                
    anomaly_maps = torch.tensor(np.stack(anomaly_maps))
    ground_truth_masks = torch.tensor(np.stack(ground_truth_masks), dtype=torch.int32)
    y_true_image = torch.tensor(y_true_image)
    image_scores = torch.tensor(image_scores)
    
    # 1. Image AUROC
    img_auroc = roc_auc_score(y_true_image.numpy(), image_scores.numpy())
    
    # 2. Confusion Matrix & Thresholding
    # Determine a simple threshold based on Youden's J statistic or percentile
    fpr, tpr, thresholds = roc_curve(y_true_image.numpy(), image_scores.numpy())
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[optimal_idx]
    
    y_pred_image = (image_scores.numpy() > optimal_threshold).astype(int)
    cm = confusion_matrix(y_true_image.numpy(), y_pred_image)
    f1 = f1_score(y_true_image.numpy(), y_pred_image)
    
    print(f"\n--- Evaluation for {category} ---")
    print(f"Image AUROC: {img_auroc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Confusion Matrix (TN, FP, FN, TP): {cm.ravel()}")
    
    # 3. AUPIMO (Strict Project Rule)
    try:
        from anomalib.data import ImageBatch
        dummy_img = torch.zeros(len(anomaly_maps), 3, SIZE, SIZE, dtype=torch.float32)
        batch = ImageBatch(image=dummy_img, anomaly_map=anomaly_maps, gt_mask=ground_truth_masks.bool())
        
        aupimo_metric = AUPIMO(num_thresholds=50000)
        aupimo_metric.update(batch)
        result = aupimo_metric.compute()
        if hasattr(result, "aupimo_scores"):
            score = float(result.aupimo_scores.nanmean().item())
        elif isinstance(result, tuple) and len(result) > 1:
            score = float(result[1].nanmean().item())
        elif isinstance(result, dict):
            score = float(next(iter(result.values())))
        else:
            score = float(result)
        print(f"AUPIMO: {score:.4f}")
    except Exception as e:
        print(f"AUPIMO Calculation Failed (Check Anomalib setup): {e}")
        
    return anomaly_maps, ground_truth_masks
    
from sklearn.metrics import roc_curve



## 6. Ablation Study
To properly isolate and compare the effectiveness of the ResNet-18 Perceptual Loss against the standard L1+SSIM loss, and to see how Masked Image Modeling (MIM) interacts with both, we will run a 2x2 grid of experiments on the `screw` category:
1. **No MIM + Standard Loss (Baseline)**
2. **No MIM + Perceptual Loss**
3. **MIM + Standard Loss**
4. **MIM + Perceptual Loss (Fully Advanced)**


In [12]:

scenarios = [
    {"name": "RUN 1 (Baseline: No MIM, No Perceptual)", "use_mim": False, "use_perceptual": False},
    {"name": "RUN 2 (Loss Test: No MIM, YES Perceptual)", "use_mim": False, "use_perceptual": True},
    {"name": "RUN 3 (MIM Test: YES MIM, No Perceptual)", "use_mim": True, "use_perceptual": False},
    {"name": "RUN 4 (Advanced: YES MIM, YES Perceptual)", "use_mim": True, "use_perceptual": True},
]

for scenario in scenarios:
    print(f"\n{'#'*60}")
    print(f"=== {scenario['name']} ===")
    print(f"{'#'*60}")
    
    model = train_autoencoder(
        "screw", 
        use_perceptual=scenario["use_perceptual"], 
        use_mim=scenario["use_mim"]
    )
    
    if model:
        evaluate_model_strict(model, "screw")




############################################################
=== RUN 1 (Baseline: No MIM, No Perceptual) ===
############################################################

Training screw | Perceptual=False | MIM=False
Epoch 01/120 - Train Loss: 0.5249 - Val Loss: 0.2604
Epoch 02/120 - Train Loss: 0.3308 - Val Loss: 0.2115
Epoch 03/120 - Train Loss: 0.3131 - Val Loss: 0.1849
Epoch 04/120 - Train Loss: 0.2734 - Val Loss: 0.1774
Epoch 05/120 - Train Loss: 0.2456 - Val Loss: 0.1577
Epoch 06/120 - Train Loss: 0.2237 - Val Loss: 0.1412
Epoch 07/120 - Train Loss: 0.2159 - Val Loss: 0.1423
Epoch 08/120 - Train Loss: 0.2076 - Val Loss: 0.1412
Epoch 09/120 - Train Loss: 0.2004 - Val Loss: 0.1292
Epoch 10/120 - Train Loss: 0.1962 - Val Loss: 0.1241
Epoch 11/120 - Train Loss: 0.1852 - Val Loss: 0.1198
Epoch 12/120 - Train Loss: 0.1784 - Val Loss: 0.1133
Epoch 13/120 - Train Loss: 0.1728 - Val Loss: 0.1280
Epoch 14/120 - Train Loss: 0.1715 - Val Loss: 0.1075
Epoch 15/120 - Train Loss: 0.1603 - Val 

Metric `AUPIMO` will save all targets and predictions in buffer. For large datasets this may lead to large memory footprint.



--- Evaluation for screw ---
Image AUROC: 0.7389
F1 Score: 0.8655
Confusion Matrix (TN, FP, FN, TP): [ 25  16  16 103]
AUPIMO: 0.0038

############################################################
=== RUN 2 (Loss Test: No MIM, YES Perceptual) ===
############################################################

Training screw | Perceptual=True | MIM=False


/home/benni/Documents/antigravity_workspace/industrial-component-anomaly-detection/.pixi/envs/dev/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 01/120 - Train Loss: 0.5392 - Val Loss: 0.2590
Epoch 02/120 - Train Loss: 0.3503 - Val Loss: 0.2295
Epoch 03/120 - Train Loss: 0.3116 - Val Loss: 0.1965
Epoch 04/120 - Train Loss: 0.2889 - Val Loss: 0.1791
Epoch 05/120 - Train Loss: 0.2694 - Val Loss: 0.1638
Epoch 06/120 - Train Loss: 0.2480 - Val Loss: 0.1560
Epoch 07/120 - Train Loss: 0.2276 - Val Loss: 0.1541
Epoch 08/120 - Train Loss: 0.2223 - Val Loss: 0.1382
Epoch 09/120 - Train Loss: 0.2148 - Val Loss: 0.1286
Epoch 10/120 - Train Loss: 0.2033 - Val Loss: 0.1550
Epoch 11/120 - Train Loss: 0.1991 - Val Loss: 0.1244
Epoch 12/120 - Train Loss: 0.1906 - Val Loss: 0.1190
Epoch 13/120 - Train Loss: 0.1814 - Val Loss: 0.1151
Epoch 14/120 - Train Loss: 0.1808 - Val Loss: 0.1102
Epoch 15/120 - Train Loss: 0.1706 - Val Loss: 0.1108
Epoch 16/120 - Train Loss: 0.1593 - Val Loss: 0.1031
Epoch 17/120 - Train Loss: 0.1486 - Val Loss: 0.1041
Epoch 18/120 - Train Loss: 0.1422 - Val Loss: 0.0905
Epoch 19/120 - Train Loss: 0.1346 - Val Loss: 

...

Metric `AUPIMO` will save all targets and predictions in buffer. For large datasets this may lead to large memory footprint.



--- Evaluation for screw ---
Image AUROC: 0.8233
F1 Score: 0.7981
Confusion Matrix (TN, FP, FN, TP): [35  6 36 83]
AUPIMO: 0.0048

############################################################
=== RUN 4 (Advanced: YES MIM, YES Perceptual) ===
############################################################

Training screw | Perceptual=True | MIM=True


/home/benni/Documents/antigravity_workspace/industrial-component-anomaly-detection/.pixi/envs/dev/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 01/120 - Train Loss: 0.5574 - Val Loss: 0.3689
Epoch 02/120 - Train Loss: 0.3379 - Val Loss: 0.2182
Epoch 03/120 - Train Loss: 0.3129 - Val Loss: 0.2321
Epoch 04/120 - Train Loss: 0.2877 - Val Loss: 0.1998
Epoch 05/120 - Train Loss: 0.2686 - Val Loss: 0.1890
Epoch 06/120 - Train Loss: 0.2650 - Val Loss: 0.1760
Epoch 07/120 - Train Loss: 0.2593 - Val Loss: 0.1616
Epoch 08/120 - Train Loss: 0.2452 - Val Loss: 0.1481
Epoch 09/120 - Train Loss: 0.2392 - Val Loss: 0.1732
Epoch 10/120 - Train Loss: 0.2313 - Val Loss: 0.1364
Epoch 11/120 - Train Loss: 0.2258 - Val Loss: 0.1328
Epoch 12/120 - Train Loss: 0.2154 - Val Loss: 0.1299
Epoch 13/120 - Train Loss: 0.2055 - Val Loss: 0.1258
Epoch 14/120 - Train Loss: 0.2039 - Val Loss: 0.1286
Epoch 15/120 - Train Loss: 0.1997 - Val Loss: 0.1182
Epoch 16/120 - Train Loss: 0.1977 - Val Loss: 0.1214
Epoch 17/120 - Train Loss: 0.1897 - Val Loss: 0.1131
Epoch 18/120 - Train Loss: 0.1870 - Val Loss: 0.1141
Epoch 19/120 - Train Loss: 0.1862 - Val Loss: 

KeyboardInterrupt: 